# **writing a function for data cleaning & standardisation purposes**

In [1]:
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.sql.types import StringType, IntegerType, DoubleType, LongType, FloatType
from functools import reduce

StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 3, Finished, Available, Finished, False)

In [2]:
# ================================================================
# SHARED UTILITY FUNCTIONS (used across all dataframes)
# ================================================================

# drop duplicates from a df based on columns input
def drop_duplicates(df, subset_cols):
    """
    drop duplicates based on a defined set of columns
    """
    return df.dropDuplicates(subset_cols)

# Trim unnecessary whitespaces from dataframe
def trim_whitespaces(df):
    """
    trim whitespaces from string columsn
    no string columns are left untouched
    """
    for col_name in df.columns:
        if dict(df.dtypes)[col_name] == "string":
            df = df.withColumn(col_name, F.trim(F.col(col_name)))
    return df

# standardise all column header to lower case
def lower_case(df):
    """
    rename all column headers to lower case
    """
    for col_name in df.columns:
        df = df.withColumnRenamed(col_name, col_name.strip().lower())
    return df

# fill nulls in descriptive columns as ""
def fill_null_desc(df, cols):
    """
    replace null values in descriptive columns with an emoty string
    """
    return df.fillna("", subset = cols)

# fill nulls in numeric columns as 0
def fill_null_num(df, cols):
    """
    replace null values in numeric columns with 0
    """
    return df.fillna(0, subset = cols)

# standardise descriptive columns to title case
def standardise_title_case(df, cols):
    """
    standardise descriptive columns to title case
    """
    for col_name in cols:
        df = df.withColumn(col_name, F.initcap(F.col(col_name)))
    return df

# remove special character from a colum
def remove_special_char(df, cols):
    pattern = r"[^a-zA-Z0-9\s.\-_]"

    for col_name in cols:
        df = df.withColumn(col_name, F.regexp_replace(F.col(col_name), pattern, ""))
    return df

# standarise date columns to proper date
def standardise_dates(df: DataFrame, col_name: str) -> DataFrame:
    stripped = F.regexp_replace(F.col(col_name), r"(?i)(st|nd|rd|th),?", "")

    date_col = F.coalesce(
        F.to_date(F.col(col_name), "yyyy/MM/dd"),
        F.to_date(F.col(col_name), "dd-MM-yyyy"),
        F.to_date(F.col(col_name), "yyyy.MM.dd"),
        F.to_date(F.col(col_name), "yyyy/MM/dd"),
        F.to_date(F.col(col_name), "yyyy-MM-dd"),
        F.to_date(F.col(col_name), "dd.MM.yyyy"),
        F.to_date(F.col(col_name), "MM/dd/yyyy"),
        F.to_date(F.col(col_name), "dd/MM/yyyy"),
        F.to_date(stripped,        "MMMM dd yyyy"),  
    )

    return df.withColumn(col_name, date_col)

# standardiseing payment type column
payment_mapping ={
    "credit_card" : "Credit Card",
    "debit_card" : "Debit Card",
    "boleto" : "Boleto",
    "voucher" : "Voucher"
}

def payment_stndising(df, col_name, mapping):
    """
    standardising the value in the payment column using a dictionary lookup
    """
    col = F.col(col_name)
    for old_value, new_value in mapping.items():
        col = F.when(F.col(col_name) == old_value, new_value).otherwise(col)
    return df.withColumn(col_name, col)

# date data qualtity flag
def date_quality_check(df, col_name):
    """
    check if a date column meets qualtity standards
    """
    date_only = F.to_date(F.col(col_name))

    stripped = F.date_format(date_only, "yyyyMMdd")

    qualtity_flag = F.when(F.length(stripped)==8, "Y").otherwise("N")

    return df.withColumn(f"{col_name}_dq", qualtity_flag)


# ================================================================
# MASTER CLEANING PIPELINES
# ================================================================



StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 4, Finished, Available, Finished, False)

In [40]:
today_date = ''

StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 42, Finished, Available, Finished, True)

In [4]:
bronze_path ='Files/bronze'
partition_path = f"processing_date={today_date}"

StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 6, Finished, Available, Finished, False)

# **Processing customer bronze df**

In [5]:
file_to_process = 'customers'

full_path = bronze_path + "/" + file_to_process + "/" + partition_path
print(full_path)


customer_bronze = spark.read.format("delta").load(full_path)


StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 7, Finished, Available, Finished, False)

Files/bronze/customers/processing_date=2026-05-01


# **cleaning & standardising customers silver**

# 1. Trim white spaces
# 2. Remove duplicates using combination of multiple columns
# 3. standardise column headers to lower case
# 4. standardise descriptive headers to title case
# 5. setting Nulls in descriptive columns as "" rather than Null

In [6]:
def clean_customers(df):
    """
    full cleaning pipeline for the customers dataframe
    """
    dedup_cols = ["customer_id", "customer_unique_id"] # this combines both column for elimination of duplicates
    descriptive_cols = ["customer_city", "customer_state"]
    title_case_cols = ["customer_city"]

    df = drop_duplicates(df, dedup_cols)
    df = lower_case(df)
    df = trim_whitespaces(df)
    df = fill_null_desc(df, descriptive_cols)
    df = standardise_title_case(df, title_case_cols)

    return df

# count df before cleaning & standardising:
bf_count_cust_bronze = customer_bronze.count()

# using function defined for cleaning & standardising df
customer_cleaned = clean_customers(customer_bronze)

# count df after cleaning & standardising:a
af_count = customer_cleaned.count()


print(f"before:{bf_count_cust_bronze} | after: {af_count} | dropped: {bf_count_cust_bronze - af_count}")


# inserting a data quality check on customer_id

customer_cleaned = customer_cleaned.withColumn(
    "customer_id_dq",
    F.when(F.col("customer_id").isNull(), "N").otherwise("Y")
)

# drop invalid row - filtering for only customer_id_dq == Y

customer_cleaned = customer_cleaned.filter(F.col("customer_id_dq") == "Y")

af_dq_count = customer_cleaned.count()

print(f"before: {af_count} | after: {af_dq_count} | dropped: {af_count - af_dq_count}")

# drop unneccasry columns
customer_cleaned = customer_cleaned.drop("processing_date", "customer_id_dq")

StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 8, Finished, Available, Finished, False)

before:1780 | after: 1780 | dropped: 0
before: 1780 | after: 1780 | dropped: 0


# **processing clean customer bronze data to silver**

In [7]:
customer_cleaned.createOrReplaceTempView("new_customer_data")

StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 9, Finished, Available, Finished, False)

# **Creating an exception handling**

In [8]:
silver_abfss_path = "abfss://f9e66737-53ce-4d4c-bfdc-31f547b675c9@onelake.dfs.fabric.microsoft.com/de8ea8e4-7996-4472-932c-105896929b23/Tables/customer_silver"

try:
    print("reading data from customer_silver")
    spark.read.format("delta").load(silver_abfss_path).createOrReplaceTempView("customer_silver")
    print("customer_silver temp view created successfully")

except:
    print("no table found, creating customer_silver table")
    spark.sql("""
        CREATE TABLE IF NOT EXISTS customer_silver (
            customer_id              STRING,
            customer_unique_id       STRING,
            customer_zip_code_prefix STRING,
            customer_city            STRING,
            customer_state           STRING
        )
        USING DELTA
    """)
    print("customer_silver table created successfully")

StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 10, Finished, Available, Finished, False)

reading data from customer_silver
no table found, creating customer_silver table
customer_silver table created successfully


# **Writing data into silver table using an UPSERT Logic**

In [9]:
sql_statement = f""" 
    MERGE INTO customer_silver AS target
    Using new_customer_data AS source
    ON target.customer_id = source.customer_id 
    AND target.customer_unique_id = source.customer_unique_id

    WHEN MATCHED THEN
    UPDATE SET    
    target.customer_zip_code_prefix = source.customer_zip_code_prefix,
    target.customer_city = source.customer_city,         
    target.customer_state = source.customer_state

    WHEN NOT MATCHED THEN
         INSERT(
         customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state) 
         VALUES (
         source.customer_id, source.customer_unique_id, source.customer_zip_code_prefix, source.customer_city, source.customer_state
         )"""

spark.sql(sql_statement).show()

StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 11, Finished, Available, Finished, False)

+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|             1780|               0|               0|             1780|
+-----------------+----------------+----------------+-----------------+



# **Processing Order payments dataframe**

In [10]:
file_to_process = 'order_payments'

full_path = bronze_path + "/" + file_to_process + "/" + partition_path
print(full_path)

order_payments_bronze = spark.read.format("delta").load(full_path)

StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 12, Finished, Available, Finished, False)

Files/bronze/order_payments/processing_date=2026-05-01


# **Cleaning & Standardising Order Payments dataframe**

# 1. Remove duplicates using combination of multiple columns
# 2. remove extra white spaces
# 3. standardise column headers, set as lower case
# 4. set numeric columns to it's appropriate data type
# 5. standard the payment_type column

In [11]:

def clean_order_payments(df):
    """
    full clean pipeline for the order_payments df
    """
    dedup_cols = ["order_id", "payment_sequential", "payment_type"] # this combines both column for elimination of duplicates
    descriptive_cols = ["payment_type"]


    df = drop_duplicates(df, dedup_cols)
    df = lower_case(df)
    df = trim_whitespaces(df)
    df = fill_null_desc(df, descriptive_cols)
    df = payment_stndising(df, "payment_type", payment_mapping)


    return df

# count df before cleaning & standardising:
bf_dup = order_payments_bronze.count()

order_payments_clean = clean_order_payments(order_payments_bronze)

# count after cleaning & standardising:

af_dup = order_payments_clean.count()

print(f"before duplicate: {bf_dup} | after duplicate: {af_dup} | dropped: {bf_dup - af_dup}")


# cast columns to appropriate data types

order_payments_clean = order_payments_clean.withColumn("payment_installments", F.col("payment_installments").cast("integer"))
order_payments_clean = order_payments_clean.withColumn("payment_sequential", F.col("payment_sequential").cast("integer"))
order_payments_clean = order_payments_clean.withColumn("payment_value", F.col("payment_value").cast("decimal(10, 2)"))

# handling Null in the column payment_type
order_payments_clean = order_payments_clean.withColumn(
    "payment_type",
    F.when(F.col("payment_type").isNull(), "n/a").otherwise(F.col("payment_type"))
)

# creating data quality check for order id
order_payments_clean = order_payments_clean.withColumn(
    "order_id_dq",
    F.when(F.col("order_id").isNotNull(), "Y").otherwise("N")
)



# drop invalid rows applying filter on order_id_dq

order_payments_clean = order_payments_clean.filter(F.col("order_id_dq") == "Y")
af_cnt = order_payments_clean.count()

print(f"before: {af_dup} | after: {af_cnt} | dropped: {af_dup - af_cnt}")

# drop uneccassary columns
order_payments_clean = order_payments_clean.drop("order_id_dq", "processing_date")

StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 13, Finished, Available, Finished, False)

before duplicate: 1886 | after duplicate: 1886 | dropped: 0
before: 1886 | after: 1886 | dropped: 0


# **processing clean order payments bronze data to silver**

In [12]:
order_payments_clean.createOrReplaceTempView("new_order_payments")

StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 14, Finished, Available, Finished, False)

# **Creating an exception handling**

In [13]:
order_payment_abfss_path = "abfss://f9e66737-53ce-4d4c-bfdc-31f547b675c9@onelake.dfs.fabric.microsoft.com/de8ea8e4-7996-4472-932c-105896929b23/Tables/order_payments_silver"

try:
    print("reading data from order_payments_silver")
    spark.read.format("delta").load(order_payment_abfss_path).createOrReplaceTempView("order_payments_silver")
    print("order_payments_silver temp view created successfully")

except:
    print("no table found, creating order_payments_silver table")
    spark.sql("""
        CREATE TABLE IF NOT EXISTS order_payments_silver (
            order_id               STRING,
            payment_sequential     INTEGER,
            payment_type            STRING,
            payment_installments   INTEGER,
            payment_value           DECIMAL(10, 2)
        )
        USING DELTA
    """)
    print("order_payments_silver table created successfully")



StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 15, Finished, Available, Finished, False)

reading data from order_payments_silver
no table found, creating order_payments_silver table
order_payments_silver table created successfully


# **Writing order payments data into silver using an UPSERT Logic**

In [14]:
sql_statement = f""" 
    MERGE INTO order_payments_silver AS target
    Using new_order_payments AS source
    ON target.order_id = source.order_id 

    WHEN MATCHED THEN
    UPDATE SET    
    target.payment_sequential = source.payment_sequential,
    target.payment_type  = source.payment_type ,         
    target.payment_installments = source.payment_installments,
    target.payment_value = source.payment_value

    WHEN NOT MATCHED THEN
         INSERT(
         order_id, payment_sequential, payment_type, payment_installments, payment_value) 
         VALUES (
         source.order_id, source.payment_sequential, source.payment_type, source.payment_installments, source.payment_value
         )"""

spark.sql(sql_statement).show()

StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 16, Finished, Available, Finished, False)

+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|             1886|               0|               0|             1886|
+-----------------+----------------+----------------+-----------------+



# **Processing Order reviews dataframe**

In [15]:
file_to_process = 'order_reviews'

full_path = bronze_path + "/" + file_to_process + "/" + partition_path
print(full_path)

order_reviews_bronze = spark.read.format("delta").load(full_path)


StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 17, Finished, Available, Finished, False)

Files/bronze/order_reviews/processing_date=2026-05-01


# **Cleaning & Standardising Order reviews dataframe**

# 1. Remove duplicates using combination of multiple columns
# 2. remove extra white spaces
# 3. standardise column headers, set as lower case
# 4. set descriptive columns with Null as ""
# 5. set numeric/date columns to it's appropriate data type
# 6. set a date data quality check

In [16]:

def clean_order_reviews(df):
    """
    full clean pipeline for the order_reviews df
    """
    dedup_cols = ["review_id", "order_id"]
    descriptive_cols = ["review_comment_title"]
    title_case_cols = ["review_comment_message"]

    df = drop_duplicates(df, dedup_cols)
    df = trim_whitespaces(df)
    df = fill_null_desc(df, descriptive_cols)
    df = standardise_title_case(df, title_case_cols)

    return df

# count df before cleaning & standardising:

before_dup = order_reviews_bronze.count()

order_reviews_cleaned = clean_order_reviews(order_reviews_bronze)

# count df after cleaning & standardising:

after_dup = order_reviews_cleaned.count()

print(f"before duplicate: {before_dup} | after duplicate: {after_dup} | dropped: {before_dup - after_dup}")


# cast columns
order_reviews_cleaned = order_reviews_cleaned.withColumn("review_score", F.col("review_score").cast("integer"))
order_reviews_cleaned = order_reviews_cleaned.withColumn("review_creation_date", F.col("review_creation_date").cast("timestamp"))
order_reviews_cleaned = order_reviews_cleaned.withColumn("review_answer_timestamp", F.col("review_answer_timestamp").cast("timestamp"))


# creating date data quality flag

order_reviews_cleaned = date_quality_check(order_reviews_cleaned, "review_creation_date")
order_reviews_cleaned = date_quality_check(order_reviews_cleaned, "review_answer_timestamp")

# check count of invalid rows
count_invalid_date_review_creation = (order_reviews_cleaned.filter(F.col("review_creation_date_dq") == "N")).count()
print(f"count of invalid rows: {count_invalid_date_review_creation}")

count_invalid_date_answer = (order_reviews_cleaned.filter(F.col("review_answer_timestamp_dq") == "N")).count()
print(f"count of invalid rows: {count_invalid_date_answer}")


# creating a data quality check for order id
order_reviews_cleaned = order_reviews_cleaned.withColumn(
    "order_id_dq",
    F.when(F.col("order_id").isNotNull(), "Y").otherwise("N")
)

# dropping rows with Null order id using dq column

order_reviews_cleaned = order_reviews_cleaned.filter(F.col("order_id_dq") == "Y")
after_count = order_reviews_cleaned.count()
print(f"before:{after_dup} | after: {after_count} | dropped: {after_dup - after_count}")

# dropping uneccessary columns:
order_reviews_cleaned = order_reviews_cleaned.drop("order_id_dq", "review_creation_date_dq", "review_answer_timestamp_dq", "processing_date")


StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 18, Finished, Available, Finished, False)

before duplicate: 1763 | after duplicate: 1763 | dropped: 0
count of invalid rows: 7
count of invalid rows: 5
before:1763 | after: 1761 | dropped: 2


# **processing clean order reviews bronze data to silver**

In [17]:
order_reviews_cleaned.createOrReplaceTempView("new_order_reviews")

StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 19, Finished, Available, Finished, False)

# **Creating an exception handling**

In [18]:
order_reviews_abfss_path = "abfss://f9e66737-53ce-4d4c-bfdc-31f547b675c9@onelake.dfs.fabric.microsoft.com/de8ea8e4-7996-4472-932c-105896929b23/Tables/order_reviews_silver"

try:
    print("reading data from order_reviews_silver")
    spark.read.format("delta").load(order_reviews_abfss_path).createOrReplaceTempView("order_reviews_silver")
    print("order_payments_silver temp view created successfully")

except:
    print("no table found, creating order_reviews_silver table")
    spark.sql("""
        CREATE TABLE IF NOT EXISTS order_reviews_silver (
            review_id               STRING,
            order_id                STRING,
            review_score            INTEGER,
            review_comment_title    STRING,
            review_comment_message  STRING,
            review_creation_date    TIMESTAMP,
            review_answer_timestamp TIMESTAMP
        )
        USING DELTA
    """)
    print("order_reviews_silver table created successfully")



StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 20, Finished, Available, Finished, False)

reading data from order_reviews_silver
no table found, creating order_reviews_silver table
order_reviews_silver table created successfully


# **writing clean order review data into Silver using an UPSERT Logic**

In [19]:
sql_statement = f""" 
    MERGE INTO order_reviews_silver AS target
    Using new_order_reviews AS source
    ON target.order_id = source.order_id AND
    target.review_id = source.review_id

    WHEN MATCHED THEN
    UPDATE SET    
    target.review_score = source.review_score,
    target.review_comment_title  = source.review_comment_title,         
    target.review_comment_message = source.review_comment_message,
    target.review_creation_date = source.review_creation_date,
    target.review_answer_timestamp = source.review_answer_timestamp

    WHEN NOT MATCHED THEN
         INSERT(
         order_id, review_id, review_score, review_comment_title, review_comment_message, review_creation_date, review_answer_timestamp) 
         VALUES (
         source.order_id, source.review_id, source.review_score, source.review_comment_title, source.review_comment_message, source.review_creation_date, source.review_answer_timestamp
         )"""

spark.sql(sql_statement).show()


StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 21, Finished, Available, Finished, False)

+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|             1761|               0|               0|             1761|
+-----------------+----------------+----------------+-----------------+



# **Processing ordered items dataframe**

In [20]:
file_to_process = 'ordered_items'

full_path = bronze_path + "/" + file_to_process + "/" + partition_path
print(full_path)

ordered_items_bronze = spark.read.format("delta").load(full_path)



StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 22, Finished, Available, Finished, False)

Files/bronze/ordered_items/processing_date=2026-05-01


# **Cleaning & Standardising Order reviews dataframe**

# 1. Remove duplicates using combination of multiple columns
# 2. remove extra white spaces
# 3. set numeric columns with Null as ""
# 5. set numeric/date columns to it's appropriate data type
# 6. set a date data quality check

In [21]:
def clean_ordered_items(df):
    """
    full clean pipeline for the ordered_item df
    """
    dedup_cols = ["order_id", "product_id", "order_item_id"]
    fillnum = ["price", "freight_value"]

    df = drop_duplicates(df, dedup_cols)
    df = trim_whitespaces(df)
    df = fill_null_num(df, fillnum)

    return df

# count df before cleaning & standardising
before_dup_orderitem = ordered_items_bronze.count()

# using function for cleaning & standardising purpose:

ordered_items_cleaned = clean_ordered_items(ordered_items_bronze)

# count df after cleaning & standardising:

after_dup_orderitem  = ordered_items_cleaned.count()

print(f"before duplicate: {before_dup_orderitem} | after duplicate: {after_dup_orderitem} | dropped: {before_dup_orderitem - after_dup_orderitem}")

# Manual clean & standardisation
# convert olumns to appropriate data type
ordered_items_cleaned = ordered_items_cleaned.withColumn("price", F.col("price").cast("decimal(10, 2)"))
ordered_items_cleaned = ordered_items_cleaned.withColumn("freight_value", F.col("freight_value").cast("decimal(10, 2)"))

# convert date column to timestamp
ordered_items_cleaned = ordered_items_cleaned.withColumn("shipping_limit_date", F.col("shipping_limit_date").cast("timestamp"))


# create a date data qualtity check for date:
ordered_items_cleaned = date_quality_check(ordered_items_cleaned, "shipping_limit_date")

# createa a data quality check for order_id

ordered_items_cleaned = ordered_items_cleaned.withColumn(
    "order_id_dq",
    F.when(F.col("order_id").isNotNull(), "Y").otherwise("N")
)


# drop columns with no order id

ordered_items_cleaned = ordered_items_cleaned.filter(F.col("order_id_dq") == "Y")
after_count_orderitem = ordered_items_cleaned.count()
print(f"before:{after_dup_orderitem} | after: {after_count_orderitem} | dropped: {after_dup_orderitem - after_count_orderitem}")

# drop uneccessary columns
ordered_items_cleaned = ordered_items_cleaned.drop("shipping_limit_date_dq", "order_id_dq", "processing_date")

 

StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 23, Finished, Available, Finished, False)

before duplicate: 1951 | after duplicate: 1951 | dropped: 0
before:1951 | after: 1951 | dropped: 0


# **processing clean ordered items data to silver**

In [22]:
ordered_items_cleaned.createOrReplaceTempView("new_ordered_items")

StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 24, Finished, Available, Finished, False)

# **Creating an exception handling**

In [23]:
silver_ordered_items_abfss_path = "abfss://f9e66737-53ce-4d4c-bfdc-31f547b675c9@onelake.dfs.fabric.microsoft.com/de8ea8e4-7996-4472-932c-105896929b23/Tables/ordered_items_silver"

try:
    print("reading data from ordered_items_silver")
    spark.read.format("delta").load(silver_ordered_items_abfss_path).createOrReplaceTempView("ordered_items_silver")
    print("ordered_items_silver temp view created successfully")

except:
    print("no table found, creating ordered_items_silver")
    spark.sql("""
        CREATE TABLE IF NOT EXISTS ordered_items_silver (
            order_id              STRING,
            order_item_id         STRING,
            product_id            STRING,
            seller_id            STRING,
            shipping_limit_date   TIMESTAMP,
            price                 DECIMAL(10, 2),
            freight_value         DECIMAL(10, 2)
        )
        USING DELTA
    """)
    print("ordered_items_silver table created successfully")

StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 25, Finished, Available, Finished, False)

reading data from ordered_items_silver
no table found, creating ordered_items_silver
ordered_items_silver table created successfully


# **Writing data into silver table using an UPSERT Logic**

In [24]:
sql_statement = f""" 
    MERGE INTO ordered_items_silver AS target
    Using new_ordered_items AS source
    ON target.order_id = source.order_id
    AND target.product_id = source.product_id
    AND target.order_item_id = source.order_item_id

    WHEN MATCHED THEN
    UPDATE SET    
    target.seller_id = source.seller_id,
    target.shipping_limit_date = source.shipping_limit_date,         
    target.price = source.price,
    target.freight_value = source.freight_value

    WHEN NOT MATCHED THEN
         INSERT(
         order_id, order_item_id, product_id, seller_id, shipping_limit_date, price, freight_value) 
         VALUES (
         source.order_id, source.order_item_id, source.product_id, source.seller_id, source.shipping_limit_date, source.price, source.freight_value
         )"""

spark.sql(sql_statement).show()

StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 26, Finished, Available, Finished, False)

+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|             1951|               0|               0|             1951|
+-----------------+----------------+----------------+-----------------+



# **Processing orders dataframe**


In [25]:
file_to_process = 'orders'

full_path = bronze_path + "/" + file_to_process + "/" + partition_path
print(full_path)

orders_bronze = spark.read.format("delta").load(full_path)


StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 27, Finished, Available, Finished, False)

Files/bronze/orders/processing_date=2026-05-01


# **Cleaning & Standardising Order reviews dataframe**

# 1. Remove duplicates using combination of multiple columns
# 2. remove extra white spaces
# 3. set numeric columns with Null as ""
# 5. set numeric/date columns to it's appropriate data type
# 6. set a date data quality check

In [26]:

def orders_clean(df):
    """
    full clean pipeline for the orders df
    """
    dedup_cols = ["order_id", "customer_id"]
    descriptive_cols = ["order_status"]
    title_case_cols = ["order_status"]

    df = drop_duplicates(df, dedup_cols)
    df = trim_whitespaces(df)
    df = fill_null_desc(df, descriptive_cols)
    df = standardise_title_case(df, title_case_cols)

    return df

# count df before cleaning & standardising:
before_dup_orders = orders_bronze.count()

# using function on df for cleaning & standardising:

cleaned_orders = orders_clean(orders_bronze)

# count df after cleaning & standardising:
after_dup_orders = cleaned_orders.count()
print(f"before duplicate: {before_dup_orders} | after duplicate: {after_dup_orders} | dropped: {before_dup_orders - after_dup_orders}")

# converting date columns to timestamp
timestamp_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"]

for col_name in timestamp_cols:
    cleaned_orders = cleaned_orders.withColumn(col_name, F.col(col_name).cast("timestamp"))

# creating a date data quality check for order purchase timestamp and order approved at
cleaned_orders = date_quality_check(cleaned_orders, "order_purchase_timestamp")
cleaned_orders = date_quality_check(cleaned_orders, "order_approved_at")
cleaned_orders = date_quality_check(cleaned_orders, "order_delivered_customer_date")

# check number of rows with invalid important date attributes?

# 1. 
print(f"count of rows before invalid row check: {cleaned_orders.count()}")
invalid_rows = (cleaned_orders.filter(F.col("order_purchase_timestamp_dq")== "Y")).count()
print(f"count of rows after invalid row check: {invalid_rows}")

# 2.
print(f"count of rows before invalid row check: {cleaned_orders.count()}")
invalid_rows2 = (cleaned_orders.filter(F.col("order_delivered_customer_date_dq")== "Y")).count()
print(f"count of rows after invalid row check: {invalid_rows2}")

# Creating a data quality flag for order_id
cleaned_orders = cleaned_orders.withColumn(
    "order_id_dq",
    F.when(F.col("order_id").isNotNull(), "Y").otherwise("N")
)


# dropping rows with Null order_id
before_count = cleaned_orders.count()
cleaned_orders = cleaned_orders.filter(F.col("order_id_dq") == "Y")
after_count = cleaned_orders.count()
print(f"before:{after_dup_orders} | after: {after_count} | dropped: {after_dup_orders - after_count}")

# drop uneccessary columns
cleaned_orders = cleaned_orders.drop("processing_date", "order_id_dq", "order_delivered_customer_date_dq", "order_approved_at_dq", "order_purchase_timestamp_dq")


StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 28, Finished, Available, Finished, False)

before duplicate: 1780 | after duplicate: 1780 | dropped: 0
count of rows before invalid row check: 1780
count of rows after invalid row check: 1780
count of rows before invalid row check: 1780
count of rows after invalid row check: 1653
before:1780 | after: 1780 | dropped: 0


# **processing clean orders bronze data to silver**

In [27]:
cleaned_orders.createOrReplaceTempView("new_orders_data")

StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 29, Finished, Available, Finished, False)

# **Creating an exception handling**

In [28]:
silver_orders_abfss_path = "abfss://f9e66737-53ce-4d4c-bfdc-31f547b675c9@onelake.dfs.fabric.microsoft.com/de8ea8e4-7996-4472-932c-105896929b23/Tables/orders_silver"

try:
    print("reading data from orders_silver")
    spark.read.format("delta").load(silver_orders_abfss_path).createOrReplaceTempView("orders_silver")
    print("orders_silver temp view created successfully")

except:
    print("no table found, creating customer_silver table")
    spark.sql("""
        CREATE TABLE IF NOT EXISTS orders_silver (
            order_id                      STRING,
            customer_id                   STRING,
            order_status                  STRING,
            order_purchase_timestamp      TIMESTAMP,
            order_approved_at             TIMESTAMP,
            order_delivered_carrier_date TIMESTAMP,
            order_delivered_customer_date TIMESTAMP,
            order_estimated_delivery_date TIMESTAMP
            )
        USING DELTA
    """)
    print("orders_silver table created successfully")

StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 30, Finished, Available, Finished, False)

reading data from orders_silver
no table found, creating customer_silver table
orders_silver table created successfully


# **Writing data into silver table using an UPSERT Logic**

In [29]:
sql_statement = f""" 
    MERGE INTO orders_silver AS target
    Using new_orders_data AS source
    ON target.order_id = source.order_id AND
    target.customer_id = source.customer_id 

    WHEN MATCHED THEN
    UPDATE SET    
    target.order_status= source.order_status,
    target.order_purchase_timestamp = source.order_purchase_timestamp,
    target.order_approved_at = source.order_approved_at,         
    target.order_delivered_carrier_date = source.order_delivered_carrier_date,
    target.order_delivered_customer_date = source.order_delivered_customer_date,         
    target.order_estimated_delivery_date = source.order_estimated_delivery_date

    WHEN NOT MATCHED THEN
         INSERT(
         order_id, customer_id, order_status, order_purchase_timestamp, order_approved_at, order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date) 
         VALUES (
         source.order_id, source.customer_id, source.order_status, source.order_purchase_timestamp, source.order_approved_at, source.order_delivered_carrier_date, source.order_delivered_customer_date, source.order_estimated_delivery_date
         )"""

spark.sql(sql_statement).show()


StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 31, Finished, Available, Finished, False)

+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|             1780|               0|               0|             1780|
+-----------------+----------------+----------------+-----------------+



# **Processing products dataframe**

In [30]:
file_to_process = 'products'

full_path = bronze_path + "/" + file_to_process + "/" + partition_path
print(full_path)

products_bronze = spark.read.format("delta").load(full_path)




StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 32, Finished, Available, Finished, False)

Files/bronze/products/processing_date=2026-05-01


# **Cleaning & Standardising products dataframe**

# 1. Remove duplicates using combination of multiple columns
# 2. remove extra white spaces
# 3. set numeric columns with Null as ""
# 5. set numeric columns to it's appropriate data type


In [31]:
# casting list of columns to decimals:

def convert_to_decimal(df, decimals=[]):
    for col_name in decimals:
        df = df.withColumn(col_name, F.col(col_name).cast("decimal(10,2)"))
    return df

products_bronze = convert_to_decimal(
  products_bronze,
  decimals=["product_name_lenght", "product_description_lenght", "product_weight_g", "product_weight_g", "product_height_cm", "product_width_cm"]  
)

# convert to integer
products_bronze = products_bronze.withColumn("product_photos_qty", F.col("product_photos_qty").cast("integer"))



# cleaning & standardising

def clean_products_bronze(df):
    """
    full clean pipeline for the products df
    """

    dedup_cols = ["product_id", "product_category_name"]
    title_case_cols = ["product_category_name"]
    numeric_cols = ["product_name_lenght", "product_description_lenght", "product_photos_qty", "product_weight_g", "product_weight_g", "product_height_cm", "product_width_cm"]

    df = drop_duplicates(df, dedup_cols)
    df = trim_whitespaces(df)
    df = standardise_title_case(df, title_case_cols)
    df = fill_null_num(df, numeric_cols)

    return df

# count before df cleaning & standardising:
before_dup_prd = products_bronze.count()

# applying data cleaning function to df
products_cleaned = clean_products_bronze(products_bronze).alias("p")

# count df after cleaning & standardising:
after_dup_prd = products_cleaned.count()

print(f"before duplicate: {before_dup_prd} | after duplicate: {after_dup_prd} | dropped: {before_dup_prd - after_dup_prd}")

products_cleaned = products_cleaned.withColumnRenamed("product_description_lenght", "product_description_length")\
.withColumnRenamed("product_name_lenght", "product_name_length")\
.withColumnRenamed("product_weight_g", "product_weight_gram")

# calling product category
product_cat = spark.read.format("csv").option("header","true").load("Files/static file/product_category_name_translation.csv").alias("pc")

# cleaning and standardising product category df
def clean_products_cat(df):
    dedup_cols = ["product_category_name", "product_category_name_english"]
    title_case_cols = ["product_category_name", "product_category_name_english"]

    df = drop_duplicates(df, dedup_cols)
    df = trim_whitespaces(df)
    df = standardise_title_case(df, title_case_cols)

    return df
product_cat = clean_products_cat(product_cat).alias("pc")


# products_cleaned = products_cleaned.alias("p")

# retrieving product_category_name_english from product category table
products_join_cleaned = products_cleaned.join(
    product_cat, 
    on = F.col("p.product_category_name") == F.col("pc.product_category_name"),
    how = "left"
).drop("product_category_name").withColumnRenamed("product_category_name_english", "product_category_name")

# create a product_id dq check
products_join_cleaned = products_join_cleaned.withColumn(
    "product_id_dq",
    F.when(F.col("product_id").isNotNull(), "Y").otherwise("N")
)

before_count = products_join_cleaned.count()
# Drop rows with Null product id

products_join_cleaned = products_join_cleaned.filter(F.col("product_id_dq")=="Y")

# count df after dropping invalid rows:
after_count_prd = products_join_cleaned.count()

print(f"before:{after_dup_prd} | after: {after_count_prd} | dropped: {after_dup_prd - after_count_prd}")


# drop uneccesary columns:
products_join_cleaned = products_join_cleaned.drop("processing_date", "product_id_dq")


StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 33, Finished, Available, Finished, False)

before duplicate: 1264 | after duplicate: 1264 | dropped: 0
before:1264 | after: 1264 | dropped: 0


# **processing clean products bronze data to silver**

In [32]:
products_join_cleaned.createOrReplaceTempView("new_products_data")

StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 34, Finished, Available, Finished, False)

# **Creating an exception handling**

In [33]:
products_silver_abfss_path = "abfss://f9e66737-53ce-4d4c-bfdc-31f547b675c9@onelake.dfs.fabric.microsoft.com/de8ea8e4-7996-4472-932c-105896929b23/Tables/products_silver"

try:
    print("reading data from products_silver")
    spark.read.format("delta").load(products_silver_abfss_path).createOrReplaceTempView("products_silver")
    print("products_silver temp view created successfully")

except:
    print("no table found, creating products_silver table")
    spark.sql("""
        CREATE TABLE IF NOT EXISTS products_silver (
            product_id                  STRING,
            product_name_length         DECIMAL(10, 2),
            product_description_length  DECIMAL(10, 2),
            product_photos_qty          DECIMAL(10, 2),
            product_weight_gram         DECIMAL(10, 2),
            product_length_cm           DECIMAL(10, 2),
            product_height_cm           DECIMAL(10, 2),
            product_width_cm            DECIMAL(10, 2),
            product_category_name       STRING
        )
        USING DELTA
    """)
    print("products_silver table created successfully")

StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 35, Finished, Available, Finished, False)

reading data from products_silver
no table found, creating products_silver table
products_silver table created successfully


# **Writing data into silver table using an UPSERT Logic**

In [34]:
sql_statement = f""" 
    MERGE INTO products_silver AS target
    Using new_products_data AS source
    ON target.product_id = source.product_id 

    WHEN MATCHED THEN
    UPDATE SET    
    target.product_name_length = source.product_name_length,
    target.product_description_length = source.product_description_length,         
    target.product_photos_qty = source.product_photos_qty,
    target.product_weight_gram  = source.product_weight_gram,         
    target.product_length_cm = source.product_length_cm,
    target.product_height_cm = source.product_height_cm,         
    target.product_width_cm = source.product_width_cm,
    target.product_category_name = source.product_category_name

    WHEN NOT MATCHED THEN
         INSERT(
         product_id , product_name_length, product_description_length, product_photos_qty, product_weight_gram, product_length_cm, product_height_cm, product_width_cm, product_category_name) 
         VALUES (
         source.product_id , source.product_name_length, source.product_description_length, source.product_photos_qty, source.product_weight_gram, source.product_length_cm, source.product_height_cm, source.product_width_cm, source.product_category_name
         )"""

spark.sql(sql_statement).show()

StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 36, Finished, Available, Finished, False)

+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|             1264|             194|               0|             1070|
+-----------------+----------------+----------------+-----------------+



# **Processing Sellers dataframe**

In [35]:
file_to_process = 'sellers'

full_path = bronze_path + "/" + file_to_process + "/" + partition_path
print(full_path)

sellers_bronze = spark.read.format("delta").load(full_path)



StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 37, Finished, Available, Finished, False)

Files/bronze/sellers/processing_date=2026-05-01


# **Cleaning & Standardising sellers dataframe**

# 1. Remove duplicates using combination of multiple columns
# 2. remove extra white spaces
# 3. set descriptive columns to title case


In [36]:

def clean_sellers(df):
    dedup_cols = ["seller_id", "seller_zip_code_prefix"]
    title_case_cols = ["seller_city"]

    df = drop_duplicates(df, dedup_cols)
    df = trim_whitespaces(df)
    df = standardise_title_case(df, title_case_cols)
    return df

# count df before cleaning & standardising:
before_dup_sellers = sellers_bronze.count()

# using function for data cleaning purposes:

cleaned_sellers = clean_sellers(sellers_bronze)

# count df after cleaning & standardising:
after_dup_sellers = cleaned_sellers.count()

print(f"before duplicate: {before_dup_sellers} | after duplicate: {after_dup_sellers} | dropped: {before_dup_sellers - after_dup_sellers}")

# drop unwanted column
cleaned_sellers = cleaned_sellers.drop("processing_date")


StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 38, Finished, Available, Finished, False)

before duplicate: 427 | after duplicate: 427 | dropped: 0


# **processing clean sellers bronze data to silver**

In [37]:
cleaned_sellers.createOrReplaceTempView("new_sellers_data")

StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 39, Finished, Available, Finished, False)

# **Creating an exception handling**

In [38]:
silver_sellers_abfss_path = "abfss://f9e66737-53ce-4d4c-bfdc-31f547b675c9@onelake.dfs.fabric.microsoft.com/de8ea8e4-7996-4472-932c-105896929b23/Tables/sellers_silver"

try:
    print("reading data from sellers_silver")
    spark.read.format("delta").load(silver_sellers_abfss_path).createOrReplaceTempView("sellers_silver")
    print("sellers_silver temp view created successfully")

except:
    print("no table found, creating sellers_silver table")
    spark.sql("""
        CREATE TABLE IF NOT EXISTS sellers_silver (
            seller_id                STRING,
            seller_zip_code_prefix   STRING,
            seller_city              STRING,
            seller_state             STRING
        )
        USING DELTA
    """)
    print("sellers_silver table created successfully")


StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 40, Finished, Available, Finished, False)

reading data from sellers_silver
no table found, creating sellers_silver table
sellers_silver table created successfully


# **Writing data into silver table using an UPSERT Logic**

In [39]:
sql_statement = f""" 
    MERGE INTO sellers_silver AS target
    Using new_sellers_data AS source
    ON target.seller_id = source.seller_id

    WHEN MATCHED THEN
    UPDATE SET    
    target.seller_zip_code_prefix = source.seller_zip_code_prefix,
    target.seller_city = source.seller_city,         
    target.seller_state = source.seller_state

    WHEN NOT MATCHED THEN
         INSERT(
         seller_id, seller_zip_code_prefix, seller_city, seller_state) 
         VALUES (
         source.seller_id, source.seller_zip_code_prefix, source.seller_city, source.seller_state
         )"""

spark.sql(sql_statement).show()

StatementMeta(, 0b8fa904-0d6f-4970-bc19-665817e50273, 41, Finished, Available, Finished, False)

+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|              427|             199|               0|              228|
+-----------------+----------------+----------------+-----------------+

